# 18 — Charge-Augmented Chemprop Screen

**Cheap screen only: a single fold, three seeds.** Does NOT run the full 25-fold CV, does NOT
build or evaluate an ensemble, does NOT prepare or send a submission.

## What this tests, and why

A D-MPNN encodes the molecular graph. A protonated basic amine and its neutral form are the
**same graph** — the current `chemprop_chemeleoninit` model has no basis to distinguish them.
Protonation state at pH 7.4 is the one candidate descriptor that is qualitatively unlike anything
the encoder can already infer (unlike molecular weight, HBA, HBD, ring count, or stereocentres,
all direct functions of the graph).

Notebook 17 established this signal empirically on this project's own curated training data, with
a threshold pre-registered before results were seen, and then confirmed it survives a
lipophilicity confound control via Spearman partial correlation:

| Isoform | charge rho | partial rho (\| logP) | Part 3/17 verdict |
|---|---|---|---|
| CYP1A2 | -0.2047 | -0.1931 | PASS |
| CYP2C9 | -0.1535 | -0.1204 | PASS |
| CYP2D6 | +0.1691 | +0.1772 | PASS |
| CYP3A4 | -0.0712 | -0.0029 | FAIL |

CYP3A4's ~96% collapse under the confound control, against the other three barely moving, is what
makes the other three credible — the control detected a genuine artifact rather than returning
SURVIVES for everything indiscriminately.

**Effect sizes are small.** No isoform cleared notebook 17's median-pIC50-difference arm; every
PASS was carried by the Spearman arm alone. A partial rho of ~0.18 corresponds to roughly a 0.17
log-unit median pIC50 gap between the +1 and 0 charge groups. This screen should be read with that
scale in mind, not as a search for a large effect.

## Two questions, answered from the SAME run — not conflated

- **Question A — solo performance.** Does the charge-augmented model beat the existing
  `chemprop_chemeleoninit` baseline on OOF ST-RAE, per isoform?
- **Question B — decorrelation.** What is this model's OOF prediction correlation against the
  existing pool members? This project's ensembling has repeatedly failed on real blind data
  (notebook 08's simple averages and notebook 12's Caruana selection both scored worse than the
  plain single model; `10c` — unensembled — remains the best base recipe). One live explanation is
  insufficient member diversity: every current pool member is a graph or fingerprint model, blind
  to protonation state in the same way. A charge-augmented model is the first pool member that
  sees something the others cannot. **A tie on Question A with genuine decorrelation on Question B
  is a useful result for this project, not a null one.**

## Scope boundaries

Does not modify `data/folds/cv_folds.csv`, `outputs/05_cv_comparison/`,
`outputs/11_caruana_prep/`, `outputs/17_protonation_charge_check/`, `src/calibration.py`, or any
existing submission artifact. Does not run the full 25-fold CV. Does not build, select, or
evaluate an ensemble — Part 3 measures decorrelation only. Does not prepare, validate, or send a
submission. Does not touch calibration or any population-moment output.

## Setup

In [1]:
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import importlib.metadata
import json

import numpy as np
import pandas as pd

from src.chemprop_screen import (
    ISOFORM_SHORT_NAMES,
    build_predict_csv,
    build_training_csv,
    load_screen_population,
    run_chemprop_predict,
    run_chemprop_train,
    run_subprocess_streamed,
    setup_logging,
    verify_predictions,
)
from src.cv_bootstrap import per_fold_bootstrap_seed
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS, REGRESSION_ENDPOINTS
from src.vendor.openadmet_eval.evaluate_predictions import add_macro_endpoint, score_activity_predictions

METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]

print(f"python: {sys.version.split()[0]}")
for pkg in ["chemprop", "torch", "numpy", "pandas"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")
print(f"REGRESSION_ENDPOINTS: {REGRESSION_ENDPOINTS}")
print(f"METRIC_NAMES: {METRIC_NAMES}")

FOLDS_PATH = REPO_ROOT / "data" / "folds" / "cv_folds.csv"
CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
CHARGE_FEATURES_PATH = REPO_ROOT / "outputs" / "17_protonation_charge_check" / "charge_logp_features.csv"
STORED_05_SCORE_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "scores" / "chemprop_chemeleoninit__repeat0_fold0.csv"

OUT = REPO_ROOT / "outputs" / "18_charge_augmented_chemprop_screen"
CHEMPROP_RUNS_DIR = OUT / "chemprop_runs"
PRED_DIR = OUT / "predictions"
SCORE_DIR = OUT / "scores"
LOG_DIR = REPO_ROOT / "logs" / "18_charge_augmented_chemprop_screen"
for d in [OUT, CHEMPROP_RUNS_DIR, PRED_DIR, SCORE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Fold choice: repeat_0, fold 0 -- named explicitly per task instructions. Chosen because it is
# the SAME (repeat, fold) notebook 05's own chemprop_chemeleoninit__repeat0_fold0 run used, which
# gives a direct, already-available reference point for Part 1's baseline sanity check.
REPEAT_COL = "repeat_0"
TEST_FOLD = 0
VAL_FRACTION = 0.15  # matches assign_screen_split's/05's own precedent

# Three seeds, shared across both arms (paired comparison, matching 05's own manifest-generation
# precedent of "one seed per computational unit, shared across the configs being compared" --
# extended here to 3 draws so seed-driven run-to-run spread is visible at all, rather than
# reusing a single global seed everywhere). Not invented fresh: drawn directly from this
# project's own already-established np.random.SeedSequence(42).spawn(25) provenance (05's own
# manifest, `scripts/generate_5x5_cv_manifest.py`) for repeat_0's folds 0/1/2 -- SEEDS[0] is
# BIT-IDENTICAL to the seed 05's own chemprop_chemeleoninit__repeat0_fold0 run used, which is
# what makes Part 1's sanity check a clean apples-to-apples comparison rather than a different
# seed entirely. Each seed drives BOTH `assign_screen_split`'s inner train/val split AND
# chemprop's own --data-seed/--pytorch-seed for that run, identically for both arms at that seed
# -- so a given (arm=baseline, seed=S) vs (arm=charge, seed=S) pair differ ONLY in whether the
# charge descriptor is present, holding the compound split and model initialization fixed.
SEEDS = [2684470948, 4091952314, 233227757]

EPOCHS = 50
PATIENCE = 5
CHEMELEON_ARCH_ARGS = ["--from-foundation", "CHEMELEON", "--multi-hot-atom-featurizer-mode", "V2"]
CHARGE_COL = "net_charge_ph74"

print(f"\nfold: {REPEAT_COL}, test_fold={TEST_FOLD}")
print(f"seeds: {SEEDS}")
print(f"epochs={EPOCHS}, patience={PATIENCE}")

python: 3.11.13
chemprop: 2.3.1
torch: 2.13.0
numpy: 1.26.4
pandas: 2.3.3
REGRESSION_ENDPOINTS: ['CYP1A2_pIC50_direct_inhibition', 'CYP2C9_pIC50_direct_inhibition', 'CYP2D6_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition']
METRIC_NAMES: ['ST-RAE', 'MAE', 'R2', 'Spearman_R', 'Kendall_Tau']

fold: repeat_0, test_fold=0
seeds: [2684470948, 4091952314, 233227757]
epochs=50, patience=5


## Pre-registered expected pattern (frozen before running anything)

Saved to `outputs/18_charge_augmented_chemprop_screen/prereg.json` **before** any training or
scoring cell below executes.

In [2]:
PREREG = {
    "notebook": "18_charge_augmented_chemprop_screen",
    "frozen_before": "any training or scoring in this notebook",
    "fold": {"repeat_col": REPEAT_COL, "test_fold": TEST_FOLD},
    "seeds": SEEDS,
    "expected_pattern": {
        "gains_plausible": [
            "CYP1A2", "CYP2C9", "CYP2D6"
        ],
        "gains_plausible_reason": (
            "these three isoforms' charge-pIC50 correlation survived notebook 17's "
            "lipophilicity confound control (partial |rho| >= 0.10, same sign as unconditional)"
        ),
        "null_control_isoform": "CYP3A4",
        "null_control_reason": (
            "CYP3A4's charge-pIC50 correlation collapsed to -0.0029 (p=0.89) under notebook 17's "
            "confound control -- there is nothing for this model to learn from the charge column "
            "on this isoform"
        ),
        "null_control_interpretation_rule": (
            "if CYP3A4 moves materially in EITHER direction, treat that as a signal that "
            "something is wrong with the plumbing (feature misalignment, row-order mismatch, "
            "leakage) rather than as a chemistry finding, and investigate before interpreting "
            "any other isoform's result"
        ),
    },
    "decorrelation_reference_point": {
        "existing_pool_pairwise_oof_correlation_range": [0.88, 0.93],
        "interpretation": (
            "existing pool members sit around 0.88-0.93 pairwise OOF correlation with each "
            "other, a range where averaging is known to buy little; meaningfully below that is "
            "the outcome of interest for Question B"
        ),
    },
    "effect_size_note": (
        "notebook 17 found small effect sizes -- no isoform cleared the median-pIC50-difference "
        "arm, every PASS was carried by the Spearman arm alone (partial rho ~0.12-0.18). This "
        "screen is read against that scale, not as a search for a large effect."
    ),
}

prereg_path = OUT / "prereg.json"
with open(prereg_path, "w") as f:
    json.dump(PREREG, f, indent=2)
print(f"saved {prereg_path}")
print(json.dumps(PREREG, indent=2))

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/prereg.json
{
  "notebook": "18_charge_augmented_chemprop_screen",
  "frozen_before": "any training or scoring in this notebook",
  "fold": {
    "repeat_col": "repeat_0",
    "test_fold": 0
  },
  "seeds": [
    2684470948,
    4091952314,
    233227757
  ],
  "expected_pattern": {
    "gains_plausible": [
      "CYP1A2",
      "CYP2C9",
      "CYP2D6"
    ],
    "gains_plausible_reason": "these three isoforms' charge-pIC50 correlation survived notebook 17's lipophilicity confound control (partial |rho| >= 0.10, same sign as unconditional)",
    "null_control_isoform": "CYP3A4",
    "null_control_reason": "CYP3A4's charge-pIC50 correlation collapsed to -0.0029 (p=0.89) under notebook 17's confound control -- there is nothing for this model to learn from the charge column on this isoform",
    "null_control_interpretation_rule": "if CYP3A4 moves materially in EITHER direction, treat that

## Part 0 — Feature preparation (no recomputation)

`net_charge_ph74` is read directly from the existing
`outputs/17_protonation_charge_check/charge_logp_features.csv` (5,655 compounds, zero compute
failures, confirmed in notebook 17) — protonation is **not** recomputed and `dimorphite-dl` is not
added to this notebook's runtime path. Only `net_charge_ph74` is used — **not** `crippen_logp`:
notebook 17's partial correlations showed charge stands on its own, and logP is the descriptor the
encoder should already be able to infer from the graph; including both would make this a
two-variable experiment and render the result unattributable to charge specifically.

In [3]:
charge_features = pd.read_csv(CHARGE_FEATURES_PATH)
print(f"loaded {CHARGE_FEATURES_PATH.name}: {charge_features.shape}")
charge_train = charge_features.loc[
    charge_features["split"] == "train", ["inchikey", CHARGE_COL]
].copy()
print(f"training-split rows: {len(charge_train)} (expected 4905)")
if len(charge_train) != 4905:
    raise ValueError(f"expected 4905 training-split charge rows, got {len(charge_train)} -- stopping.")
if charge_train[CHARGE_COL].isna().any():
    raise ValueError("charge_logp_features.csv has null net_charge_ph74 on a training row -- stopping.")
if charge_train["inchikey"].duplicated().any():
    raise ValueError("charge_logp_features.csv has duplicate training inchikeys -- stopping.")

loaded charge_logp_features.csv: (5655, 7)
training-split rows: 4905 (expected 4905)


### Chemprop's native extra-descriptor (X_d) mechanism — confirmed against the installed API

Checked directly against the installed `chemprop==2.3.1` CLI (`chemprop train --help` /
`chemprop predict --help`) and its source (`chemprop/cli/train.py`) before writing any training
code:

- `--descriptors-columns <col ...>` — column name(s) **in the same input CSV** carrying extra
  molecule-level ("datapoint") descriptors, concatenated to the aggregated post-message-passing
  representation before the prediction FFN. This is the exact mechanism this task specifies, and
  it changes nothing else about the architecture — no per-task heads, no custom wrapper. It is
  used here in preference to the alternative `--descriptors-path` (a separate file, aligned to the
  main input by row **position**): keeping the descriptor in the same CSV row as its own SMILES
  and targets removes any possibility of a position-based misalignment between files.
- **Descriptor scaling** (`train.py`, `normalize_inputs`): with `--no-descriptor-scaling` **not**
  passed (the default), chemprop fits a `StandardScaler` on `train_dset`'s `X_d` values only —
  confirmed directly from source: `scaler = train_dset.normalize_inputs("X_d")`, then
  `val_dset.normalize_inputs("X_d", scaler)` (applied, not refit, to the validation split). In
  this project's split terminology that means the scaler is fit on `screen_inner_train` rows only,
  **excluding both `screen_inner_val` and `screen_test`** — stricter than just excluding the
  held-out fold, since it also never lets the early-stopping validation slice inform the scaler
  (standard practice: validation data should never inform any fit, including a feature scaler).
  This is read as satisfying, and being stricter than, this task's "training-fold statistics
  only, not train+heldout together" requirement. The fitted scaler is baked into the saved model
  checkpoint (`_model.X_d_transform`) and applied automatically and identically to `screen_test`
  compounds' **raw** descriptor values at predict time via `chemprop predict --model-paths ...` —
  so the same raw `net_charge_ph74` integer column is written to both the training and prediction
  CSVs; no manual pre-standardization is performed, and `--no-descriptor-scaling` is **not**
  passed to either `train` or `predict`. Chemprop's own fitted `loc`/`scale` (printed to its own
  log at train time) is cross-checked below against an independently pandas-computed mean/std over
  the same `screen_inner_train` population, as a correctness check rather than an assumption.

In [4]:
def build_training_csv_with_charge(population, target_cols, descriptor_col, out_path, logger, require_all_targets):
    # Mirrors src/chemprop_screen.py's build_training_csv exactly, plus one extra descriptor
    # column in the written CSV -- the shared module's own function is reused unmodified for the
    # BASELINE arm; this local variant exists only because that function's column list is
    # hardcoded and has no hook for an extra column.
    pooled = population[population["screen_split"].isin(["screen_inner_train", "screen_inner_val"])].copy()
    n_before = len(pooled)
    if require_all_targets:
        pooled = pooled.dropna(subset=target_cols)
    logger.info(
        f"{out_path.name}: pooled-train rows before label filter={n_before}, "
        f"after={len(pooled)} (require_all_targets={require_all_targets}, targets={target_cols}, "
        f"descriptor_col={descriptor_col})"
    )
    split_map = {"screen_inner_train": "train", "screen_inner_val": "val"}
    pooled["chemprop_split"] = pooled["screen_split"].map(split_map)
    if pooled["chemprop_split"].isna().any():
        raise ValueError(f"unexpected screen_split value(s) building {out_path.name} -- stopping.")
    cols = ["canonical_smiles", *target_cols, descriptor_col, "chemprop_split"]
    pooled[cols].to_csv(out_path, index=False)
    logger.info(
        f"wrote {out_path} ({len(pooled)} rows, chemprop_split counts: "
        f"{pooled['chemprop_split'].value_counts().to_dict()})"
    )
    return pooled


def build_predict_csv_with_charge(population, descriptor_col, chemprop_runs_dir, logger):
    chemprop_runs_dir.mkdir(parents=True, exist_ok=True)
    test_df = population.loc[
        population["screen_split"] == "screen_test",
        ["Molecule_Name", "inchikey", "canonical_smiles", descriptor_col],
    ].copy()
    out_path = chemprop_runs_dir / "predict_input.csv"
    test_df.to_csv(out_path, index=False)
    logger.info(f"wrote {out_path} ({len(test_df)} screen_test compounds, with {descriptor_col})")
    return out_path


def run_chemprop_predict_with_descriptor(model_dir, predict_csv, descriptor_col, output_csv, logger):
    # Mirrors src/chemprop_screen.py's run_chemprop_predict exactly, plus --descriptors-columns
    # -- that function's argv is hardcoded with no hook for extra flags, so a local variant is
    # used here rather than modifying the shared module. Still reuses run_subprocess_streamed
    # (the actual non-trivial shared machinery: the stdin fix and streamed logging) unmodified.
    argv = [
        "chemprop", "predict",
        "-i", str(predict_csv),
        "-s", "canonical_smiles",
        "--descriptors-columns", descriptor_col,
        "--model-paths", str(model_dir),
        "--accelerator", "cpu",
        "--devices", "1",
        "-o", str(output_csv),
    ]
    run_subprocess_streamed(argv, logger)


print("defined build_training_csv_with_charge, build_predict_csv_with_charge, "
      "run_chemprop_predict_with_descriptor")

defined build_training_csv_with_charge, build_predict_csv_with_charge, run_chemprop_predict_with_descriptor


### Build the fold's population once per seed, join charge, verify alignment

`load_screen_population` (unmodified, `src/chemprop_screen.py`) is called once per seed —
`screen_test` membership is identical across all three calls (it depends only on
`cv_folds["repeat_0"] == 0`, not on `seed`; asserted below), while the `screen_inner_train`/
`screen_inner_val` split varies per seed, matching how a real "rerun with a different seed" would
behave. The charge column is merged onto this SAME population dataframe that both arms' CSV
builders read from — since the descriptor travels as a column of that one dataframe (not a
separately-aligned file), there is no position-based row-order risk to check beyond confirming the
merge introduced no nulls.

In [5]:
setup_log = setup_logging(LOG_DIR / "setup.log", "nb18_setup")

populations = {}
screen_test_inchikey_sets = []
for seed in SEEDS:
    population = load_screen_population(
        FOLDS_PATH, CURATED_PATH, REPEAT_COL, TEST_FOLD, VAL_FRACTION, seed, setup_log
    )
    n_before = len(population)
    population = population.merge(charge_train, on="inchikey", how="left")
    n_after = len(population)
    n_missing_charge = int(population[CHARGE_COL].isna().sum())
    print(f"seed={seed}: population rows before charge join={n_before}, after={n_after}, "
          f"missing charge={n_missing_charge}")
    if n_after != n_before or n_missing_charge:
        raise ValueError(
            f"seed={seed}: charge join changed row count or left nulls -- stopping before "
            "training anything on a misaligned population."
        )
    populations[seed] = population
    screen_test_inchikey_sets.append(
        set(population.loc[population["screen_split"] == "screen_test", "inchikey"])
    )

# screen_test membership must be IDENTICAL across all three seeds (test_fold assignment does not
# depend on seed) -- confirmed explicitly rather than assumed.
if not all(s == screen_test_inchikey_sets[0] for s in screen_test_inchikey_sets[1:]):
    raise ValueError("screen_test compound set differs across seeds -- stopping, this should be impossible.")
print(f"\nscreen_test compound count (identical across all 3 seeds): {len(screen_test_inchikey_sets[0])}")

for seed, population in populations.items():
    test_df = population[population["screen_split"] == "screen_test"]
    print(f"seed={seed} screen_test net_charge_ph74: n={len(test_df)}, "
          f"n_null={int(test_df[CHARGE_COL].isna().sum())}")

2026-09-14 22:45:14,277 [INFO] loaded cv_folds.csv: (4905, 7)


2026-09-14 22:45:14,277 [INFO] loaded train_inhibition_curated.csv: (4905, 20)


2026-09-14 22:45:14,284 [INFO] assign_screen_split(repeat_col='repeat_0', test_fold=0, val_fraction=0.15, seed=2684470948) split counts: {'screen_inner_train': 3335, 'screen_test': 981, 'screen_inner_val': 589}


2026-09-14 22:45:14,286 [INFO] rows after SMILES/target join: 4905 (missing: 0)


2026-09-14 22:45:14,287 [INFO]   CYP1A2_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 975, 'screen_inner_val': 160, 'screen_test': 277}


...

2026-09-14 22:45:14,330 [INFO]   CYP1A2_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 965, 'screen_inner_val': 170, 'screen_test': 277}


2026-09-14 22:45:14,331 [INFO]   CYP2C9_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 864, 'screen_inner_val': 157, 'screen_test': 264}


2026-09-14 22:45:14,331 [INFO]   CYP2D6_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 1041, 'screen_inner_val': 157, 'screen_test': 295}


2026-09-14 22:45:14,332 [INFO]   CYP3A4_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 1587, 'screen_inner_val': 298, 'screen_test': 450}


seed=233227757: population rows before charge join=4905, after=4905, missing charge=0

screen_test compound count (identical across all 3 seeds): 981
seed=2684470948 screen_test net_charge_ph74: n=981, n_null=0
seed=4091952314 screen_test net_charge_ph74: n=981, n_null=0
seed=233227757 screen_test net_charge_ph74: n=981, n_null=0


## Part 1 — Training

Two arms, both `chemprop_chemeleoninit`, everything identical (architecture, `--epochs 50
--patience 5`, learning rate, batch size, all four isoforms in one multitask model) except the
CHARGE arm's one extra `--descriptors-columns net_charge_ph74`. The BASELINE arm is retrained here
(via the unmodified `build_training_csv`/`build_predict_csv`/`run_chemprop_train`/
`run_chemprop_predict` from `src/chemprop_screen.py`) rather than reusing
`outputs/05_cv_comparison/`'s stored numbers, so both arms share the exact same fold, seeds,
environment, and Chemprop version — that file is read afterward for reference only, never
modified.

In [6]:
def run_one_arm_seed(arm, seed, population):
    tag = f"{arm}__seed{seed}"
    cached_pred_path = PRED_DIR / f"{tag}.csv"
    if cached_pred_path.exists():
        # Matches this project's own is_done()-checkpointing convention
        # (scripts/run_5x5_cv_comparison.py) -- lets this notebook be re-executed cheaply
        # (e.g. after fixing a downstream analysis bug) without redoing real training.
        cached = pd.read_csv(cached_pred_path)
        expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
        if set(cached["Molecule_Name"]) == expected_names and len(cached) == len(expected_names):
            print(f"  [cached] {tag}: reusing {cached_pred_path}")
            return cached
        print(f"  [cache mismatch] {tag}: cached file doesn't match this population -- retraining.")

    run_dir = CHEMPROP_RUNS_DIR / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logging(LOG_DIR / f"{tag}.log", f"nb18_{tag}")

    train_csv = run_dir / "train_input.csv"
    if arm == "baseline":
        build_training_csv(population, REGRESSION_ENDPOINTS, train_csv, logger, require_all_targets=False)
        predict_csv = build_predict_csv(population, run_dir, logger)
        arch_args = CHEMELEON_ARCH_ARGS
    elif arm == "charge":
        build_training_csv_with_charge(
            population, REGRESSION_ENDPOINTS, CHARGE_COL, train_csv, logger, require_all_targets=False
        )
        predict_csv = build_predict_csv_with_charge(population, CHARGE_COL, run_dir, logger)
        arch_args = CHEMELEON_ARCH_ARGS + ["--descriptors-columns", CHARGE_COL]
    else:
        raise ValueError(f"unknown arm: {arm}")

    run_chemprop_train(train_csv, REGRESSION_ENDPOINTS, run_dir, logger, arch_args, EPOCHS, PATIENCE, seed)

    raw_pred_csv = run_dir / "raw_predictions.csv"
    if arm == "baseline":
        run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)
    else:
        run_chemprop_predict_with_descriptor(run_dir / "model_0", predict_csv, CHARGE_COL, raw_pred_csv, logger)

    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
    verify_predictions(raw_pred_csv, expected_names, REGRESSION_ENDPOINTS, logger)

    pred_df = pd.read_csv(raw_pred_csv)
    pred_df.to_csv(PRED_DIR / f"{tag}.csv", index=False)
    return pred_df


RUN_ORDER = [("baseline", s) for s in SEEDS] + [("charge", s) for s in SEEDS]
print("run order:", RUN_ORDER)

run order: [('baseline', 2684470948), ('baseline', 4091952314), ('baseline', 233227757), ('charge', 2684470948), ('charge', 4091952314), ('charge', 233227757)]


In [7]:
raw_predictions = {}
timings = {}

arm0, seed0 = RUN_ORDER[0]
print(f"=== FIRST RUN: arm={arm0}, seed={seed0} -- timing this one explicitly before launching the rest ===")
t0 = time.time()
raw_predictions[(arm0, seed0)] = run_one_arm_seed(arm0, seed0, populations[seed0])
t1 = time.time()
timings[(arm0, seed0)] = t1 - t0
print(f"\nFIRST RUN ({arm0}, seed={seed0}) took {t1 - t0:.1f}s ({(t1 - t0) / 60:.2f} min)")
print(f"rough budget estimate for the remaining 5 runs at this pace: "
      f"~{5 * (t1 - t0) / 60:.1f} min (actual per-run time varies with early stopping)")

=== FIRST RUN: arm=baseline, seed=2684470948 -- timing this one explicitly before launching the rest ===
  [cached] baseline__seed2684470948: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/predictions/baseline__seed2684470948.csv

FIRST RUN (baseline, seed=2684470948) took 0.0s (0.00 min)
rough budget estimate for the remaining 5 runs at this pace: ~0.0 min (actual per-run time varies with early stopping)


In [8]:
for arm, seed in RUN_ORDER[1:]:
    print(f"\n=== arm={arm}, seed={seed} ===")
    t0 = time.time()
    raw_predictions[(arm, seed)] = run_one_arm_seed(arm, seed, populations[seed])
    t1 = time.time()
    timings[(arm, seed)] = t1 - t0
    print(f"({arm}, seed={seed}) took {t1 - t0:.1f}s ({(t1 - t0) / 60:.2f} min)")

total_min = sum(timings.values()) / 60
print(f"\ntotal training+predict time across all {len(RUN_ORDER)} runs: {total_min:.2f} min")
for (arm, seed), secs in timings.items():
    print(f"  {arm} seed={seed}: {secs:.1f}s")


=== arm=baseline, seed=4091952314 ===
  [cached] baseline__seed4091952314: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/predictions/baseline__seed4091952314.csv
(baseline, seed=4091952314) took 0.0s (0.00 min)

=== arm=baseline, seed=233227757 ===
  [cached] baseline__seed233227757: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/predictions/baseline__seed233227757.csv
(baseline, seed=233227757) took 0.0s (0.00 min)

=== arm=charge, seed=2684470948 ===
  [cached] charge__seed2684470948: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/predictions/charge__seed2684470948.csv
(charge, seed=2684470948) took 0.0s (0.00 min)

=== arm=charge, seed=4091952314 ===
  [cached] charge__seed4091952314: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/predictions/charge__seed4091952314

### Cross-check: Chemprop's own fitted descriptor scaler vs. an independent pandas computation

Confirms Part 0's stated mechanism actually behaved as described (train-split-only fit), rather
than assuming the source-code read was correct.

In [9]:
charge_seed0_log = LOG_DIR / f"charge__seed{seed0}.log"
scaler_lines = [
    line for line in charge_seed0_log.read_text().splitlines() if "Descriptors: loc" in line
]
print(f"chemprop's own reported descriptor scaler (charge arm, seed={seed0}):")
for line in scaler_lines:
    print(f"  {line}")

train_only_mask = (
    (populations[seed0]["screen_split"] == "screen_inner_train")
)
independent_mean = populations[seed0].loc[train_only_mask, CHARGE_COL].mean()
independent_std = populations[seed0].loc[train_only_mask, CHARGE_COL].std(ddof=0)
print(f"\nindependently computed (pandas, screen_inner_train only, seed={seed0}): "
      f"mean={independent_mean:.6f}, std(ddof=0)={independent_std:.6f}")
print("(ddof=0 population std, matching sklearn's StandardScaler convention used internally by chemprop)")

chemprop's own reported descriptor scaler (charge arm, seed=2684470948):
  2026-09-14 22:12:59,145 [INFO] 2026-09-14T22:12:59 - INFO:chemprop.cli.train - Descriptors: loc = [0.37], scale = [0.839]

independently computed (pandas, screen_inner_train only, seed=2684470948): mean=0.370015, std(ddof=0)=0.839326
(ddof=0 population std, matching sklearn's StandardScaler convention used internally by chemprop)


## Part 2 — Question A: solo performance

Held-out predictions scored with the vendored, unmodified evaluator
(`src/vendor/openadmet_eval/`), same call pattern as `scripts/run_5x5_cv_comparison.py` (each
run's own seed reused as its bootstrap seed, matching that script's established one-seed-per-unit
convention).

In [10]:
curated = pd.read_csv(CURATED_PATH)

def score_run(arm, seed, pred_df):
    ground_truth = curated[curated["inchikey"].isin(pred_df["inchikey"])].copy()
    with per_fold_bootstrap_seed(seed):
        scored = score_activity_predictions(pred_df, ground_truth, REGRESSION_ENDPOINTS)
        scored = add_macro_endpoint(scored, REGRESSION_ENDPOINTS, ACTIVITY_METRICS)
    scored["arm"] = arm
    scored["seed"] = seed
    scored.to_csv(SCORE_DIR / f"{arm}__seed{seed}.csv", index=False)
    return scored

scores = {}
for (arm, seed), pred_df in raw_predictions.items():
    scores[(arm, seed)] = score_run(arm, seed, pred_df)
    print(f"scored {arm} seed={seed}: {len(scores[(arm, seed)])} bootstrap rows")

point_rows = []
for (arm, seed), scored in scores.items():
    pe = scored.groupby("Endpoint")[METRIC_NAMES].mean()
    for endpoint in pe.index:
        row = {"arm": arm, "seed": seed, "Endpoint": endpoint}
        row.update(pe.loc[endpoint].to_dict())
        point_rows.append(row)
point_df = pd.DataFrame(point_rows)
point_df.to_csv(OUT / "point_estimates.csv", index=False)
print(f"\nsaved {OUT / 'point_estimates.csv'} ({len(point_df)} rows)")
print(point_df.to_string(index=False))

2026-09-14 22:45:14.377 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-14 22:45:14.378 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-09-14 22:45:14.379 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-14 22:45:14.379 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


2026-09-14 22:45:14.973 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP2C9_pIC50_direct_inhibition


...

2026-09-14 22:45:29.093 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric MAE across bootstrap iterations


2026-09-14 22:45:29.094 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-14 22:45:29.094 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-14 22:45:29.095 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


scored charge seed=233227757: 5000 bootstrap rows

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/18_charge_augmented_chemprop_screen/point_estimates.csv (30 rows)
     arm       seed                       Endpoint   ST-RAE      MAE       R2  Spearman_R  Kendall_Tau
baseline 2684470948 CYP1A2_pIC50_direct_inhibition 0.788388 0.659005 0.294865    0.527503     0.371380
baseline 2684470948 CYP2C9_pIC50_direct_inhibition 0.711301 0.527769 0.360901    0.583438     0.411169
baseline 2684470948 CYP2D6_pIC50_direct_inhibition 0.958037 0.598534 0.164962    0.417043     0.282962
baseline 2684470948 CYP3A4_pIC50_direct_inhibition 0.491035 0.529702 0.605361    0.794581     0.597336
baseline 2684470948                             MA 0.737190 0.578753 0.356522    0.580641     0.415712
baseline 4091952314 CYP1A2_pIC50_direct_inhibition 0.815870 0.663525 0.270579    0.516736     0.365741
baseline 4091952314 CYP2C9_pIC50_direct_inhibition 0.731190 0.535882 0.337484    0.551962     0.39

### Baseline sanity check: retrained baseline (seed matching 05's own) vs. 05's stored numbers

`SEEDS[0]` (2684470948) is bit-identical to the seed `outputs/05_cv_comparison`'s own
`chemprop_chemeleoninit__repeat0_fold0` run used for this exact fold — same data, same
architecture, same environment. This is read-only against that file; nothing in
`outputs/05_cv_comparison/` is modified.

In [11]:
stored_05 = pd.read_csv(STORED_05_SCORE_PATH)
stored_pe = stored_05.groupby("Endpoint")[METRIC_NAMES].mean()

retrained_pe = point_df[(point_df["arm"] == "baseline") & (point_df["seed"] == seed0)].set_index("Endpoint")

print(f"{'Endpoint':35s} {'stored 05 ST-RAE':>18s} {'retrained ST-RAE':>18s} {'abs diff':>10s}")
max_diff = 0.0
for endpoint in REGRESSION_ENDPOINTS:
    stored_val = stored_pe.loc[endpoint, "ST-RAE"]
    retrained_val = retrained_pe.loc[endpoint, "ST-RAE"]
    diff = abs(stored_val - retrained_val)
    max_diff = max(max_diff, diff)
    print(f"{endpoint:35s} {stored_val:18.4f} {retrained_val:18.4f} {diff:10.4f}")

DISCREPANCY_FLAG_THRESHOLD = 0.03  # stated explicitly: comparable to the seed-to-seed spread magnitude seen below
print(f"\nmax abs ST-RAE difference: {max_diff:.4f} (flag threshold: {DISCREPANCY_FLAG_THRESHOLD})")
if max_diff > DISCREPANCY_FLAG_THRESHOLD:
    print("FLAG: retrained baseline differs from 05's stored numbers by more than the flag "
          "threshold on at least one isoform -- investigate before trusting the charge-arm "
          "comparison below.")
else:
    print("OK: retrained baseline matches 05's stored numbers closely -- proceeding.")

Endpoint                              stored 05 ST-RAE   retrained ST-RAE   abs diff
CYP1A2_pIC50_direct_inhibition                  0.7884             0.7884     0.0000
CYP2C9_pIC50_direct_inhibition                  0.7113             0.7113     0.0000
CYP2D6_pIC50_direct_inhibition                  0.9580             0.9580     0.0000
CYP3A4_pIC50_direct_inhibition                  0.4910             0.4910     0.0000

max abs ST-RAE difference: 0.0000 (flag threshold: 0.03)
OK: retrained baseline matches 05's stored numbers closely -- proceeding.


### ST-RAE per isoform per arm: mean ± spread across the three seeds

In [12]:
summary_rows = []
for endpoint in REGRESSION_ENDPOINTS:
    iso = endpoint.split("_")[0]
    for arm in ["baseline", "charge"]:
        vals = point_df.loc[
            (point_df["Endpoint"] == endpoint) & (point_df["arm"] == arm), "ST-RAE"
        ].to_numpy()
        summary_rows.append({
            "isoform": iso, "arm": arm, "n_seeds": len(vals),
            "st_rae_mean": vals.mean(), "st_rae_std": vals.std(ddof=1),
            "st_rae_min": vals.min(), "st_rae_max": vals.max(),
        })
st_rae_summary = pd.DataFrame(summary_rows)
st_rae_summary.to_csv(OUT / "st_rae_summary.csv", index=False)
print(st_rae_summary.to_string(index=False))

isoform      arm  n_seeds  st_rae_mean  st_rae_std  st_rae_min  st_rae_max
 CYP1A2 baseline        3     0.799546    0.014451    0.788388    0.815870
 CYP1A2   charge        3     0.812654    0.014969    0.801541    0.829675
 CYP2C9 baseline        3     0.714179    0.015770    0.700046    0.731190
 CYP2C9   charge        3     0.714873    0.012074    0.704450    0.728103
 CYP2D6 baseline        3     0.959941    0.062582    0.898334    1.023454
 CYP2D6   charge        3     0.963789    0.059074    0.897942    1.012134
 CYP3A4 baseline        3     0.522321    0.031277    0.491035    0.553590
 CYP3A4   charge        3     0.507886    0.012284    0.494104    0.517680


In [13]:
def solo_verdict_for(endpoint):
    iso = endpoint.split("_")[0]
    base_row = st_rae_summary[(st_rae_summary["isoform"] == iso) & (st_rae_summary["arm"] == "baseline")].iloc[0]
    charge_row = st_rae_summary[(st_rae_summary["isoform"] == iso) & (st_rae_summary["arm"] == "charge")].iloc[0]
    diff = charge_row["st_rae_mean"] - base_row["st_rae_mean"]  # negative = charge arm better (lower ST-RAE)
    combined_spread = max(base_row["st_rae_std"], charge_row["st_rae_std"])
    resolved = abs(diff) > combined_spread
    if not resolved:
        verdict = "tie (unresolved -- diff smaller than within-arm seed spread)"
    elif diff < 0:
        verdict = "CHARGE improves"
    else:
        verdict = "CHARGE loses"
    return {
        "baseline_mean": base_row["st_rae_mean"], "charge_mean": charge_row["st_rae_mean"],
        "diff": diff, "combined_seed_spread": combined_spread, "resolved": resolved, "verdict": verdict,
    }


print("=== CYP3A4 control check (checked FIRST, per the pre-registered rule, before interpreting "
      "the other three isoforms) ===\n")
cyp3a4_endpoint = [e for e in REGRESSION_ENDPOINTS if e.startswith("CYP3A4")][0]
cyp3a4 = solo_verdict_for(cyp3a4_endpoint)
print(f"CYP3A4: baseline={cyp3a4['baseline_mean']:.4f}, charge={cyp3a4['charge_mean']:.4f}, "
      f"diff={cyp3a4['diff']:+.4f}, combined_seed_spread={cyp3a4['combined_seed_spread']:.4f} "
      f"-> {cyp3a4['verdict']}")
print()
if cyp3a4["resolved"]:
    print("CYP3A4 CONTROL: moved by a RESOLVED amount (larger than seed spread) -- per the "
          "pre-registered interpretation rule, this is a signal something is wrong with the "
          "plumbing (feature misalignment, row-order mismatch, leakage), not a chemistry "
          "finding. STOPPING here rather than interpreting the other three isoforms until this "
          "is investigated.")
    raise RuntimeError(
        "CYP3A4 within-experiment control moved by a resolved amount -- investigate plumbing "
        "before interpreting CYP1A2/CYP2C9/CYP2D6. See the printed diff/spread above."
    )
else:
    print("CYP3A4 CONTROL: unresolved (within seed spread), as pre-registered -- consistent with "
          "there being nothing for the model to learn from the charge column on this isoform. "
          "Proceeding to interpret the other three isoforms.\n")

print("=== all four isoforms (CYP3A4 repeated for a complete table) ===\n")
verdicts_solo = {}
for endpoint in REGRESSION_ENDPOINTS:
    iso = endpoint.split("_")[0]
    verdicts_solo[iso] = cyp3a4 if iso == "CYP3A4" else solo_verdict_for(endpoint)
    r = verdicts_solo[iso]
    marker = " <-- WITHIN-EXPERIMENT CONTROL (already checked above)" if iso == "CYP3A4" else ""
    print(f"{iso}{marker}: baseline={r['baseline_mean']:.4f}, charge={r['charge_mean']:.4f}, "
          f"diff={r['diff']:+.4f}, combined_seed_spread={r['combined_seed_spread']:.4f} -> {r['verdict']}")

=== CYP3A4 control check (checked FIRST, per the pre-registered rule, before interpreting the other three isoforms) ===

CYP3A4: baseline=0.5223, charge=0.5079, diff=-0.0144, combined_seed_spread=0.0313 -> tie (unresolved -- diff smaller than within-arm seed spread)

CYP3A4 CONTROL: unresolved (within seed spread), as pre-registered -- consistent with there being nothing for the model to learn from the charge column on this isoform. Proceeding to interpret the other three isoforms.

=== all four isoforms (CYP3A4 repeated for a complete table) ===

CYP1A2: baseline=0.7995, charge=0.8127, diff=+0.0131, combined_seed_spread=0.0150 -> tie (unresolved -- diff smaller than within-arm seed spread)
CYP2C9: baseline=0.7142, charge=0.7149, diff=+0.0007, combined_seed_spread=0.0158 -> tie (unresolved -- diff smaller than within-arm seed spread)
CYP2D6: baseline=0.9599, charge=0.9638, diff=+0.0038, combined_seed_spread=0.0626 -> tie (unresolved -- diff smaller than within-arm seed spread)
CYP3A4 <

## Part 3 — Question B: decorrelation

`outputs/11_caruana_prep/oof_long_{iso}.csv` is read-only here. Filtered to `repeat==0,
fold==0` — the SAME held-out compounds as this notebook's own `screen_test` (asserted below,
not assumed: `screen_test` membership is a pure function of `cv_folds["repeat_0"] == 0`, which is
exactly how notebook 05's own chemprop runs constructed their `repeat==0, fold==0` held-out set
too). The CHARGE and BASELINE arms' three seeds are averaged per compound into one representative
prediction each (analogous to a deep-ensemble mean), with the per-seed range reported alongside
as a robustness check on that averaging.

In [14]:
def seed_averaged_predictions(arm):
    frames = [raw_predictions[(arm, seed)].set_index("inchikey")[REGRESSION_ENDPOINTS] for seed in SEEDS]
    avg = sum(frames) / len(frames)
    return avg.reset_index()

charge_avg = seed_averaged_predictions("charge")
baseline_avg = seed_averaged_predictions("baseline")
print(f"charge_avg: {charge_avg.shape}, baseline_avg: {baseline_avg.shape}")

POOL_MEMBER_COLS = [
    "chemeleon__lightgbm", "chemeleon__rf", "chemeleon__xgboost",
    "chemprop_chemeleoninit", "chemprop_randominit",
    "ecfp4_narrow__lightgbm", "ecfp4_narrow__rf", "ecfp4_narrow__xgboost",
    "mordred_pca__lightgbm", "mordred_pca__rf", "mordred_pca__xgboost",
]

correlation_matrices = {}
error_correlation_results = {}
for endpoint in REGRESSION_ENDPOINTS:
    iso = endpoint.split("_")[0]
    pool = pd.read_csv(REPO_ROOT / "outputs" / "11_caruana_prep" / f"oof_long_{iso}.csv")
    pool_fold = pool[(pool["repeat"] == 0) & (pool["fold"] == TEST_FOLD)].copy()

    charge_iso = charge_avg[["inchikey", endpoint]].rename(columns={endpoint: "charge_avg"})
    baseline_iso = baseline_avg[["inchikey", endpoint]].rename(columns={endpoint: "baseline_avg"})
    merged = pool_fold.merge(charge_iso, on="inchikey", how="inner").merge(baseline_iso, on="inchikey", how="inner")

    if len(merged) != len(pool_fold):
        raise ValueError(
            f"{iso}: {len(pool_fold)} pool rows but only {len(merged)} matched this notebook's "
            "own screen_test predictions by inchikey -- stopping, fold mismatch suspected."
        )
    print(f"{iso}: n={len(merged)} (pool repeat=0,fold={TEST_FOLD} rows, all matched)")

    corr_cols = POOL_MEMBER_COLS + ["charge_avg", "baseline_avg"]
    corr_matrix = merged[corr_cols].corr(method="pearson")
    correlation_matrices[iso] = corr_matrix
    corr_matrix.to_csv(OUT / f"correlation_matrix_{iso}.csv")

    merged["charge_error"] = merged["charge_avg"] - merged["y_true"]
    merged["baseline_error"] = merged["baseline_avg"] - merged["y_true"]
    error_corr = merged["charge_error"].corr(merged["baseline_error"])
    pred_corr_vs_baseline = merged["charge_avg"].corr(merged["baseline_avg"])
    pool_pairwise = merged[POOL_MEMBER_COLS].corr(method="pearson")
    pool_pairwise_vals = pool_pairwise.to_numpy()
    pool_pairwise_mean = pool_pairwise_vals[np.triu_indices_from(pool_pairwise_vals, k=1)].mean()
    charge_vs_pool_mean = corr_matrix.loc["charge_avg", POOL_MEMBER_COLS].mean()

    error_correlation_results[iso] = {
        "n": len(merged),
        "charge_vs_baseline_prediction_corr": pred_corr_vs_baseline,
        "charge_vs_baseline_error_corr": error_corr,
        "charge_vs_pool_mean_correlation": charge_vs_pool_mean,
        "pool_pairwise_mean_correlation": pool_pairwise_mean,
    }
    print(f"  charge vs pool mean correlation: {charge_vs_pool_mean:.4f} "
          f"(existing pool's own pairwise mean: {pool_pairwise_mean:.4f})")
    print(f"  charge vs baseline: prediction corr={pred_corr_vs_baseline:.4f}, "
          f"ERROR corr={error_corr:.4f}")

charge_avg: (981, 5), baseline_avg: (981, 5)
CYP1A2: n=277 (pool repeat=0,fold=0 rows, all matched)
  charge vs pool mean correlation: 0.6666 (existing pool's own pairwise mean: 0.5640)
  charge vs baseline: prediction corr=0.9763, ERROR corr=0.9872
CYP2C9: n=264 (pool repeat=0,fold=0 rows, all matched)
  charge vs pool mean correlation: 0.6933 (existing pool's own pairwise mean: 0.6042)
  charge vs baseline: prediction corr=0.9834, ERROR corr=0.9901
CYP2D6: n=295 (pool repeat=0,fold=0 rows, all matched)
  charge vs pool mean correlation: 0.6316 (existing pool's own pairwise mean: 0.5019)
  charge vs baseline: prediction corr=0.9633, ERROR corr=0.9893
CYP3A4: n=450 (pool repeat=0,fold=0 rows, all matched)
  charge vs pool mean correlation: 0.8144 (existing pool's own pairwise mean: 0.7706)
  charge vs baseline: prediction corr=0.9899, ERROR corr=0.9837


### Full correlation matrices, per isoform

In [15]:
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
for iso, matrix in correlation_matrices.items():
    print(f"\n=== {iso} ===")
    print(matrix.round(3).to_string())


=== CYP1A2 ===
                        chemeleon__lightgbm  chemeleon__rf  chemeleon__xgboost  chemprop_chemeleoninit  chemprop_randominit  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  charge_avg  baseline_avg
chemeleon__lightgbm                   1.000          0.845               0.659                   0.715                0.626                   0.622             0.612                  0.487                  0.643            0.618                 0.447       0.745         0.735
chemeleon__rf                         0.845          1.000               0.702                   0.670                0.618                   0.610             0.638                  0.482                  0.646            0.680                 0.479       0.748         0.719
chemeleon__xgboost                    0.659          0.702               1.000                   0.586                0.515                   0.505      

### Per-seed robustness check: CHARGE arm's correlation with `chemprop_chemeleoninit`, per seed

In [16]:
for endpoint in REGRESSION_ENDPOINTS:
    iso = endpoint.split("_")[0]
    pool = pd.read_csv(REPO_ROOT / "outputs" / "11_caruana_prep" / f"oof_long_{iso}.csv")
    pool_fold = pool[(pool["repeat"] == 0) & (pool["fold"] == TEST_FOLD)][["inchikey", "chemprop_chemeleoninit"]]
    per_seed_corrs = []
    for seed in SEEDS:
        charge_seed_df = raw_predictions[("charge", seed)][["inchikey", endpoint]]
        merged_seed = pool_fold.merge(charge_seed_df, on="inchikey", how="inner")
        per_seed_corrs.append(merged_seed["chemprop_chemeleoninit"].corr(merged_seed[endpoint]))
    print(f"{iso}: per-seed charge-vs-chemprop_chemeleoninit correlation = "
          f"{[round(c, 4) for c in per_seed_corrs]}, "
          f"seed-averaged (used above) = {correlation_matrices[iso].loc['charge_avg', 'chemprop_chemeleoninit']:.4f}")

CYP1A2: per-seed charge-vs-chemprop_chemeleoninit correlation = [0.8742, 0.8839, 0.8723], seed-averaged (used above) = 0.9280
CYP2C9: per-seed charge-vs-chemprop_chemeleoninit correlation = [0.9265, 0.8879, 0.8982], seed-averaged (used above) = 0.9427
CYP2D6: per-seed charge-vs-chemprop_chemeleoninit correlation = [0.8487, 0.7466, 0.8224], seed-averaged (used above) = 0.8625
CYP3A4: per-seed charge-vs-chemprop_chemeleoninit correlation = [0.9521, 0.9311, 0.9056], seed-averaged (used above) = 0.9538


In [17]:
error_corr_df = pd.DataFrame(error_correlation_results).T
error_corr_df.to_csv(OUT / "decorrelation_summary.csv")
print(error_corr_df.to_string())

            n  charge_vs_baseline_prediction_corr  charge_vs_baseline_error_corr  charge_vs_pool_mean_correlation  pool_pairwise_mean_correlation
CYP1A2  277.0                            0.976283                       0.987203                         0.666616                        0.564039
CYP2C9  264.0                            0.983399                       0.990118                         0.693280                        0.604180
CYP2D6  295.0                            0.963309                       0.989335                         0.631555                        0.501887
CYP3A4  450.0                            0.989889                       0.983678                         0.814428                        0.770643


## Part 4 — Verdict (report, do not act)

### The pre-registered decorrelation reference point does not match what this fold actually shows — corrected here, not silently

The pre-registration (`prereg.json`) stated existing pool members "sit around 0.88-0.93 pairwise
OOF correlation with each other," and treated meaningfully-below-that as the outcome of interest.
**The pool's own pairwise correlation actually computed on this single fold's 277-450 held-out
compounds is much lower than that — 0.50 to 0.77 depending on isoform** (`decorrelation_summary.csv`,
`pool_pairwise_mean_correlation`), not 0.88-0.93. The most likely explanation: this project's own
prior correlation work (notebook 11b) computed **residual** (prediction − true) correlation
pooled across all 25 CV folds (~1,400+ compounds), not raw-prediction correlation on one
~300-compound fold — a single fold's narrower compound range and smaller n plausibly produces a
lower raw-prediction correlation estimate than a 25-fold-pooled, wider-range, residual-based one;
this is a plausible account, not a confirmed one, and is stated as such.

Given this, applying the pre-registered **absolute** 0.85 cutoff to `charge_vs_pool_mean_correlation`
alone — as an earlier version of this cell did — is misleading: on that absolute cutoff, every
isoform reads as "decorrelated," but **`charge_vs_pool_mean_correlation` is actually HIGHER than
`pool_pairwise_mean_correlation` in all four isoforms** (confirmed directly, not assumed:
`decorrelation_summary.csv`) — the charge arm is, if anything, *slightly more* correlated with
the existing pool than the pool's own members are with each other, the opposite of "meaningfully
decorrelated." The verdict below uses the correct, apples-to-apples comparison instead: the
charge arm's mean correlation with the pool **relative to** the pool's own internal mean,
computed identically on this same fold — not an absolute number carried over from a mismatched
reference population.

In [18]:
final_verdicts = {}
for endpoint in REGRESSION_ENDPOINTS:
    iso = endpoint.split("_")[0]
    solo = verdicts_solo[iso]
    decorr = error_correlation_results[iso]

    solo_improves = solo["resolved"] and solo["diff"] < 0
    solo_ties = not solo["resolved"]
    solo_loses = solo["resolved"] and solo["diff"] > 0
    # Corrected verdict: decorrelated only if the charge arm's mean correlation with the pool
    # sits BELOW the pool's own internal pairwise mean, computed on this same fold (see markdown
    # above) -- not an absolute cutoff borrowed from a reference population this fold doesn't match.
    is_decorrelated = decorr["charge_vs_pool_mean_correlation"] < decorr["pool_pairwise_mean_correlation"]

    if solo_improves and is_decorrelated:
        outcome = "(a) solo improves AND decorrelated -> candidate for both direct use and the ensemble pool"
    elif solo_ties and is_decorrelated:
        outcome = "(b) solo ties BUT decorrelated -> not a solo deployment candidate, but a real ensemble-pool candidate"
    elif solo_ties and not is_decorrelated:
        outcome = "(c) solo ties AND correlated at pool-typical levels -> charge signal not reaching the output in a way that matters"
    elif solo_loses and is_decorrelated:
        outcome = "solo LOSES but decorrelated -> not a solo candidate; ensemble-pool value undetermined by this screen alone"
    else:
        outcome = "solo LOSES and correlated at pool-typical levels -> no case for this mechanism on this isoform"

    full_25_fold_candidate = solo_improves or (solo_ties and is_decorrelated)

    final_verdicts[iso] = {
        "solo_verdict": solo["verdict"],
        "decorrelated": bool(is_decorrelated),
        "charge_vs_pool_mean_correlation": decorr["charge_vs_pool_mean_correlation"],
        "outcome": outcome,
        "full_25_fold_cv_candidate": bool(full_25_fold_candidate),
    }
    print(f"\n{iso}:")
    print(f"  solo: {solo['verdict']} (diff={solo['diff']:+.4f}, seed spread={solo['combined_seed_spread']:.4f})")
    print(f"  decorrelation: charge-vs-pool mean corr={decorr['charge_vs_pool_mean_correlation']:.4f} "
          f"(pool's own pairwise mean={decorr['pool_pairwise_mean_correlation']:.4f}) "
          f"-> {'DECORRELATED' if is_decorrelated else 'pool-typical'}")
    print(f"  {outcome}")
    print(f"  full-25-fold-CV candidate on this screen's evidence: {full_25_fold_candidate} "
          f"(NOT run in this task)")

verdict_path = OUT / "verdict.json"
with open(verdict_path, "w") as f:
    json.dump(final_verdicts, f, indent=2, default=float)
print(f"\nsaved {verdict_path}")


CYP1A2:
  solo: tie (unresolved -- diff smaller than within-arm seed spread) (diff=+0.0131, seed spread=0.0150)
  decorrelation: charge-vs-pool mean corr=0.6666 (pool's own pairwise mean=0.5640) -> pool-typical
  (c) solo ties AND correlated at pool-typical levels -> charge signal not reaching the output in a way that matters
  full-25-fold-CV candidate on this screen's evidence: False (NOT run in this task)

CYP2C9:
  solo: tie (unresolved -- diff smaller than within-arm seed spread) (diff=+0.0007, seed spread=0.0158)
  decorrelation: charge-vs-pool mean corr=0.6933 (pool's own pairwise mean=0.6042) -> pool-typical
  (c) solo ties AND correlated at pool-typical levels -> charge signal not reaching the output in a way that matters
  full-25-fold-CV candidate on this screen's evidence: False (NOT run in this task)

CYP2D6:
  solo: tie (unresolved -- diff smaller than within-arm seed spread) (diff=+0.0038, seed spread=0.0626)
  decorrelation: charge-vs-pool mean corr=0.6316 (pool's own 

### Summary, stated plainly — no cross-isoform averaging

**Question A (solo performance): a clean tie on all four isoforms.** No isoform's charge-arm vs.
baseline-arm ST-RAE difference exceeds its own combined 3-seed spread — CYP1A2 (+0.0131 vs. spread
0.0150), CYP2C9 (+0.0007 vs. 0.0158), CYP2D6 (+0.0038 vs. 0.0626), CYP3A4 (-0.0144 vs. 0.0313, the
control). Every difference is unresolved, including on the three isoforms notebook 17 flagged as
plausible. The CYP3A4 control landed exactly as pre-registered (unresolved), so the plumbing is
trusted and this is read as a real result, not an artifact.

**Question B (decorrelation): also negative, once corrected.** An analysis bug was caught and
fixed during this notebook's own review (see the markdown directly above the verdict cell): the
pre-registered 0.88-0.93 reference point does not match this fold's actual pool-internal
correlation (0.50-0.77), and an earlier version of the verdict cell that compared against that
mismatched absolute number wrongly labelled every isoform "decorrelated." Corrected to the
apples-to-apples comparison — the charge arm's mean correlation with the pool relative to the
pool's own internal mean, both computed on this same fold — the charge arm sits at **pool-typical
levels on all four isoforms**, slightly above the pool's own internal average in every case, not
below it. The charge arm's error correlation with its own baseline specifically is 0.984-0.990 —
about as correlated as two independently-seeded instances of the same plain recipe would be.

**Verdict: (c) on all four isoforms** — solo ties AND correlated at pool-typical levels. The
charge signal notebook 17 established in the raw descriptor does not appear to be reaching this
model's output in a way that matters, at least via this one mechanism (a single shared scalar
`X_d` column, one fold, three seeds). **No isoform is a candidate for the full 25-fold CV
comparison on this screen's evidence.** Per this task's scope, this is a report, not an action: no
model is built, retrained, or submitted here, and this result does not itself prove the mechanism
can never work — only that this specific, minimal implementation does not show a benefit on this
screen. A materially different approach (e.g. per-task rather than shared descriptor injection,
which this task deliberately did not test) would need its own evidence before further investment.